# 04 — Testing & scenario comparison (Step 4)
Compare **NN vs RF** across the closest-users scenarios — `X ∈ {3,5,10}` × encoding (`pos`/`agg`) plus the
**X=0 no-neighbour baseline** (users sampled at random from the full ACC Arena population) — on
MSE / MAE / R² and training duration, and identify the best configuration.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # time granularity: THE dataset-size knob (professor's guidance: coarsen the
                                 # time grid rather than subsample users). Raw cadence is ~3s with frequent 4s
                                 # steps, duplicate stamps and gaps up to 7s (ACC) / ~1s (Salt&Tar); keep >= 10
                                 # so the nearest-alignment tolerance stays above the worst raw gap.
N_USERS          = None          # ACC Arena users. None = ALL (~12k), so the X-closest neighbourhoods are the
                                 # real ones; an int (e.g. 1500, seeded random sample) biases the neighbourhoods
                                 # (X closest searched within the sample) and is only for quick debug runs.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
                                 # X=0 (no neighbour features) IS produced and trained: it is the baseline
                                 # that quantifies what the closest-users features actually add.
ENCODINGS        = ["pos", "agg"]# neighbour-feature encodings: "pos" = ordered per-neighbour columns
                                 # (nb0_*, nb1_*, ...), "agg" = order-invariant aggregates over the X
                                 # neighbours (sum/mean/count) — rationale in notebook 02.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
BEST_ENC         = "pos"         # encoding used for the transfer-learning experiment (pick from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): at full population the top ~1% of active samples carry
                                 # ~86% of the total variance -> MSE/R2 would be about a handful of rare peaks.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
res = pd.read_csv(f"{RESULTS_DIR}/metrics.csv")
if "enc" not in res.columns:                  # metrics.csv from a run before the encoding experiment
    res["enc"] = "pos"
res.sort_values(["model","enc","X"])

## Metrics vs X, per encoding
Solid = positional encoding, dashed = aggregated encoding, dotted horizontal = **X=0 no-neighbour baseline**.
The vertical gap between a curve and its baseline is the *net contribution of the closest-users features*.

In [ ]:
metrics = ["MSE","MAE","R2","train_s"] + (["infer_ms"] if "infer_ms" in res.columns else [])
colors = {"NN": "tab:blue", "RF": "tab:orange"}
styles = {"pos": "-", "agg": "--"}
fig, ax = plt.subplots(1, len(metrics), figsize=(4*len(metrics),4))
for i, metric in enumerate(metrics):
    for model in ["NN","RF"]:
        for enc in [e for e in ["pos","agg"] if e in res.enc.values]:
            d = res[(res.model==model) & (res.enc==enc)].sort_values("X")
            ax[i].plot(d.X, d[metric], marker="o", color=colors[model], linestyle=styles[enc],
                       label=f"{model} {enc}")
        base = res[(res.model==model) & (res.enc=="none")]
        if len(base):                          # X=0 baseline: no neighbour features at all
            ax[i].axhline(base[metric].iloc[0], color=colors[model], linestyle=":", alpha=.7,
                          label=f"{model} X=0")
    ax[i].set_xlabel("X (closest users)"); ax[i].set_title(metric); ax[i].legend(fontsize=8); ax[i].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_metrics_vs_X.png", dpi=120); plt.show()

## Slide-ready summary: R² per scenario
The five-panel strip above is for analysis; this single panel is the presentation figure — every scenario on
one axis, both models, baseline as the reference line.

In [ ]:
GREEN, RED, INK, MUT = "#2a9d8f", "#e76f51", "#333333", "#777777"   # project palette

scen = [("X=0 baseline", res[res.enc == "none"])]
for x in sorted(res.X[res.X > 0].unique()):
    for enc, lbl in [("pos", "positional"), ("agg", "aggregated")]:
        d = res[(res.X == x) & (res.enc == enc)]
        if len(d):
            scen.append((f"X={x} {lbl}", d))

fig, ax = plt.subplots(figsize=(8, 0.62 * len(scen) + 1.6))
ypos = np.arange(len(scen))[::-1]
for model, color, off in [("NN", GREEN, .13), ("RF", RED, -.13)]:
    xs, ys = [], []
    for y, (_, d) in zip(ypos, scen):
        v = d.loc[d.model == model, "R2"]
        if len(v):
            xs.append(float(v.iloc[0])); ys.append(y + off)
    ax.scatter(xs, ys, s=70, color=color, zorder=3, edgecolor="white", lw=1, label=model)
    for x_, y_ in zip(xs, ys):
        ax.text(x_ + .003, y_, f"{x_:.3f}", va="center", fontsize=8.5, color=INK)
base = res[res.enc == "none"]
if len(base):
    ax.axvline(base.R2.max(), color=MUT, lw=1, ls=":", zorder=1)
ax.set_yticks(ypos, [n for n, _ in scen])
ax.set_xlabel("R\u00b2 (test)")
ax.set_title("Do the X-closest-users features help? Every scenario vs the no-neighbour baseline", pad=30)
ax.legend(frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0), labelcolor=INK)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=.25); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_r2_summary.png", dpi=150); plt.show()

## Best configuration and pred-vs-true

In [ ]:
best = res.loc[res.R2.idxmax()]
print("Best:", best[["model","X","enc","MSE","MAE","R2","train_s"]].to_dict())

import numpy as np, joblib
from tensorflow import keras
x, enc = int(best.X), best.enc
sfx = "_agg" if enc == "agg" else ""
d = np.load(f"{PROCESSED_DIR}/acc_X{x}{sfx}.npz"); Xte, yte = d["X_test"], d["y_test"]
if best.model == "NN":
    pred = keras.models.load_model(f"{RESULTS_DIR}/models/nn_X{x}{sfx}.keras").predict(Xte, verbose=0).ravel()
else:
    pred = joblib.load(f"{RESULTS_DIR}/models/rf_X{x}{sfx}.pkl").predict(Xte)

plt.figure(figsize=(6, 5))
lim = [0, max(yte.max(), pred.max())]
hb = plt.hexbin(yte, pred, gridsize=60, cmap="Greens", mincnt=1, extent=(*lim, *lim))
plt.plot(lim, lim, color="#e76f51", lw=1.5, ls="--")
plt.colorbar(hb, label="samples")
plt.xlabel("True throughput (Mbps)"); plt.ylabel("Predicted (Mbps)")
plt.title(f"{best.model} (X={x}, {enc}) \u2014 R\u00b2={best.R2:.3f}, MAE={best.MAE:.2f} Mbps")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_pred_vs_true.png", dpi=150); plt.show()

## Residual analysis (best model)
Two standard diagnostics: residual vs predicted (systematic under/over-prediction, heteroscedasticity) and the
residual distribution. Expected here: residuals centred at zero with variance growing with the predicted value
(heavy-tailed target), and a long **negative** tail — the model under-predicts the rare bursts (pred − true
< 0 when the true value spikes), consistent with demand being unobservable.

In [ ]:
res_ = pred - yte
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
hb = ax[0].hexbin(pred, res_, gridsize=60, cmap="Greens", mincnt=1)
ax[0].axhline(0, color=INK, lw=1)
ax[0].set_xlabel("predicted (Mbps)"); ax[0].set_ylabel("residual = pred \u2212 true (Mbps)")
ax[0].set_title("Residuals vs predicted")
fig.colorbar(hb, ax=ax[0], label="samples")
ax[1].hist(res_, bins=80, color=GREEN)
ax[1].axvline(0, color=INK, lw=1, ls="--")
ax[1].set_xlabel("residual (Mbps)"); ax[1].set_ylabel("samples")
ax[1].set_title(f"Residual distribution \u2014 MAE={np.abs(res_).mean():.2f}, "
                f"RMSE={np.sqrt((res_**2).mean()):.2f} Mbps")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
    a.grid(alpha=.25); a.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_residuals.png", dpi=150); plt.show()

## Error slices: where does the model err?
Aggregate metrics hide structure. Left: MAE per traffic class (recovered from the one-hot columns of the test
matrix). Right: MAE per decile of the true throughput — errors grow with the target's magnitude, i.e. the
model is accurate in the typical regime and loses the (rare) high-demand samples.

In [ ]:
import json as _json
sfx_b = "_agg" if enc == "agg" else ""
cols_b = _json.load(open(f"{PROCESSED_DIR}/acc_X{x}{sfx_b}_cols.json"))
Xte_b = d["X_test"]
tt_idx = [i for i, c in enumerate(cols_b) if c.startswith("traffic_")]
tt_cls = np.array([int(cols_b[i].split("_")[1]) for i in tt_idx])
tt = tt_cls[np.argmax(Xte_b[:, tt_idx], axis=1)]
tlab = {2: "const", 3: "video", 4: "gaming", 5: "http"}
abs_err = np.abs(pred - yte)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
classes = [t for t in sorted(tlab) if (tt == t).any()]
mae_c = [abs_err[tt == t].mean() for t in classes]
bars = ax[0].bar(range(len(classes)), mae_c, color=GREEN, width=0.55)
for b, v in zip(bars, mae_c):
    ax[0].text(b.get_x() + b.get_width()/2, v + .01, f"{v:.2f}", ha="center", color=INK, fontsize=9)
ax[0].set_xticks(range(len(classes)), [tlab[t] for t in classes])
ax[0].set_ylabel("MAE (Mbps)"); ax[0].set_title("MAE per traffic class")

qs = np.quantile(yte, np.linspace(0, 1, 11))
bin_id = np.clip(np.searchsorted(qs, yte, side="right") - 1, 0, 9)
filled = [b for b in range(10) if (bin_id == b).any()]   # duplicate-heavy targets can leave empty deciles
centers = [np.median(yte[bin_id == b]) for b in filled]
mae_d = [abs_err[bin_id == b].mean() for b in filled]
ax[1].plot(centers, mae_d, marker="o", ms=6, lw=2, color=GREEN)
ax[1].set_xlabel("true throughput, decile median (Mbps)"); ax[1].set_ylabel("MAE (Mbps)")
ax[1].set_title("MAE per true-throughput decile")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
    a.grid(alpha=.25); a.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_error_slices.png", dpi=150); plt.show()

## How much do the closest-users features contribute? RF feature importance
Side-by-side for the two encodings of the mid scenario: with the **positional** encoding the co-located
neighbours are interchangeable, so their importance is diluted over many `nbK_*` columns; the **aggregated**
encoding concentrates the same information in a handful of order-invariant columns (`nb_prb_sum` ≈ cell load).

In [ ]:
import json as _json
x_imp = 5                                     # scenario to inspect
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for a, sfx, title in [(axes[0], "", "positional (nbK_*)"), (axes[1], "_agg", "aggregated (nb_*)")]:
    rf = joblib.load(f"{RESULTS_DIR}/models/rf_X{x_imp}{sfx}.pkl")
    cols = _json.load(open(f"{PROCESSED_DIR}/acc_X{x_imp}{sfx}_cols.json"))
    imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
    nb_mask = imp.index.str.startswith("nb")
    print(f"{title:22s} own features {imp[~nb_mask].sum()*100:.1f}%  |  "
          f"neighbour features {imp[nb_mask].sum()*100:.1f}%  (spread over {nb_mask.sum()} columns)")
    top = imp.head(15)[::-1]
    top.plot.barh(ax=a, color=["#e76f51" if c.startswith("nb") else "#2a9d8f" for c in top.index])
    a.set_title(f"RF importance, X={x_imp}, {title}")
fig.suptitle("green = own features, red = neighbour features", y=1.02)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_feature_importance.png", dpi=120,
                                bbox_inches="tight"); plt.show()

## Permutation importance (rigorous check on the impurity ranking)
Impurity-based importance is computed on the *training* structure and dilutes correlated features (exactly our
co-located neighbours). Permutation importance measures the actual drop in test R² when a feature is shuffled
— the deployment-relevant notion. Computed on a test subsample for speed; error bars = std over repeats.

In [ ]:
from sklearn.inspection import permutation_importance
x_pi, sfx_pi = x_imp, "_agg"                       # the aggregated scenario: few, interpretable columns
rf_pi = joblib.load(f"{RESULTS_DIR}/models/rf_X{x_pi}{sfx_pi}.pkl")
cols_pi = _json.load(open(f"{PROCESSED_DIR}/acc_X{x_pi}{sfx_pi}_cols.json"))
d_pi = np.load(f"{PROCESSED_DIR}/acc_X{x_pi}{sfx_pi}.npz")
sub = np.random.default_rng(RANDOM_SEED).choice(len(d_pi["X_test"]), min(20000, len(d_pi["X_test"])), replace=False)
pi = permutation_importance(rf_pi, d_pi["X_test"][sub], d_pi["y_test"][sub],
                            n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1, scoring="r2")

order = np.argsort(pi.importances_mean)[-12:]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
imp_gini = pd.Series(rf_pi.feature_importances_, index=cols_pi).sort_values().tail(12)
ax[0].barh(range(len(imp_gini)), imp_gini.values,
           color=[RED if c.startswith("nb") else GREEN for c in imp_gini.index])
ax[0].set_yticks(range(len(imp_gini)), imp_gini.index)
ax[0].set_title(f"Impurity importance (train structure), X={x_pi} agg")
ax[1].barh(range(len(order)), pi.importances_mean[order], xerr=pi.importances_std[order],
           color=[RED if cols_pi[i].startswith("nb") else GREEN for i in order], error_kw=dict(ecolor=INK, lw=1))
ax[1].set_yticks(range(len(order)), [cols_pi[i] for i in order])
ax[1].set_title(f"Permutation importance (test R\u00b2 drop), X={x_pi} agg")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
    a.grid(axis="x", alpha=.25); a.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_perm_importance.png", dpi=150); plt.show()

**Interpretation notes (context from the raw-data analysis).** Users are placed on discrete spots and heavily
co-located (at a given instant, tens of users can share the exact same position), and throughput in this
dataset is strongly **demand-driven**: the user's own `prb` dominates, while neighbour columns share the
remainder. Reading guide:
- **X=0 baseline vs the curves**: the gap is the net value of the closest-users features. If a curve sits on
  the baseline, that scenario's neighbour features add nothing.
- **pos vs agg**: with co-located (tied) neighbours the positional ordering is arbitrary, so `nbK_*` columns
  are permutation noise and the RF dilutes their importance; the aggregates (`nb_prb_sum` ≈ cell load,
  `nb_active_count`) encode the contention mechanism directly and should use fewer, stronger columns.
- **X growth**: check whether R² improves with X or the extra columns only dilute the RF's per-split feature
  sampling (`max_features="sqrt"`), and weigh the training-cost increase.
Users are sampled **at random from the whole venue**, so the neighbour sample is unbiased w.r.t. user ids.

## Takeaways
- Compare how the metrics move across `X ∈ {3,5,10}` **and across encodings (pos vs agg)** for both models,
  always against the **X=0 no-neighbour baseline**.
- Note the **accuracy vs training-cost** trade-off (training duration grows with X / model complexity; the
  aggregated encoding keeps the feature count constant in X).
- Set `BEST_X` **and `BEST_ENC`** in the config of notebook **05** to the best-performing combination for
  the transfer-learning study.

## Presentation recap — the X experiment at a glance
Four slide-ready views of the whole X study (they only re-read `res`, nothing above is recomputed):
**(a)** the anatomy of the feature vector per scenario — what "features from the X closest users" actually
means, and why the schema is frozen (`BEST_X`, `BEST_ENC`) before transfer learning;
**(b)** one table with R², MAE and training time for every scenario × model, best per column highlighted;
**(c)** quality-vs-cost scatter — each configuration as one point, the winner must sit top-left;
**(d)** NN-vs-RF dumbbell per scenario — who wins where, and by how much.

In [ ]:
# === Recap (a): feature-vector anatomy — what exactly enters the model, per scenario ===
# Every scenario shares the same 13 base columns (7 own numeric + 6 traffic one-hot); the X-closest
# feature enters as 5 columns PER neighbour (dist, sinr_dl, sinr_ul, prb, bler) in the positional
# encoding, or as 5 order-invariant aggregates IN TOTAL in the aggregated one. The fixed schema is what
# makes the TL notebook possible: the pretrained input layer's width must match column-by-column.
from matplotlib.patches import Patch

BASE_N, TRAF_N, PER_NB = 7, 6, 5
scen_feat = [("X=0 baseline", 0)] + [(f"X={x_} positional", x_ * PER_NB) for x_ in sorted(X_VALUES)] \
            + [(f"X={'/'.join(map(str, sorted(X_VALUES)))} aggregated", PER_NB)]
BLUE = "#457b9d"
fig, ax = plt.subplots(figsize=(9.5, 0.7 * len(scen_feat) + 1))
ypos = np.arange(len(scen_feat))[::-1]
for y, (name, nb_n) in zip(ypos, scen_feat):
    left = 0
    for w, color in [(BASE_N, GREEN), (TRAF_N, BLUE), (nb_n, RED)]:
        if w:
            ax.barh(y, w, left=left, height=.58, color=color); left += w
    ax.text(left + .8, y, f"{left} features", va="center", color=INK, fontweight="bold")
ax.set_yticks(ypos); ax.set_yticklabels([n for n, _ in scen_feat])
ax.legend(handles=[Patch(color=GREEN, label="own numeric (7: bler, prb, sinr_dl/ul, x, y, z)"),
                   Patch(color=BLUE, label="traffic one-hot (6 classes)"),
                   Patch(color=RED, label="neighbour features (5 per neighbour pos, 5 total agg)")],
          frameon=False, loc="lower right", fontsize=8.5)
ax.set_xlabel("features in the model input")
ax.set_title("Feature-vector anatomy: positional grows by 5 per neighbour, aggregated stays constant")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=.25); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_feature_schema.png", dpi=150); plt.show()

In [ ]:
# === Recap (b): all scenarios in one table — best value per column highlighted ===
# One slide-ready table: R², MAE and training time per scenario for both models. Bold + tinted cell =
# best of the column. This is where "is R² alone enough?" gets answered visually: if the R² winner and
# the MAE winner coincide, the model choice is robust to the metric.
scen_t = [("X=0 baseline", res[res.enc == "none"])]
for x_ in sorted(res.X[res.X > 0].unique()):
    for enc_, lbl in [("pos", "positional"), ("agg", "aggregated")]:
        d_ = res[(res.X == x_) & (res.enc == enc_)]
        if len(d_): scen_t.append((f"X={x_} {lbl}", d_))

cols_t = ["R² NN", "R² RF", "MAE NN", "MAE RF", "train s NN", "train s RF"]
data, names = [], []
for name, d_ in scen_t:
    row = []
    for metric in ["R2", "MAE", "train_s"]:
        for model in ["NN", "RF"]:
            v = d_.loc[d_.model == model, metric]
            row.append(float(v.iloc[0]) if len(v) else np.nan)
    data.append(row); names.append(name)
tab = pd.DataFrame(data, index=names, columns=cols_t)
best_of = {c: (tab[c].idxmax() if c.startswith("R²") else tab[c].idxmin()) for c in cols_t}

fig, ax = plt.subplots(figsize=(9.5, 0.42 * len(tab) + 1.2)); ax.axis("off")
cell_text = [[f"{tab.loc[n, c]:.3f}" if "train" not in c else f"{tab.loc[n, c]:.0f}"
              for c in cols_t] for n in tab.index]
t = ax.table(cellText=cell_text, rowLabels=list(tab.index), colLabels=cols_t,
             loc="center", cellLoc="center")
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.35)
for (r_, c_), cell in t.get_celld().items():
    cell.set_edgecolor("#dddddd")
    if r_ == 0:
        cell.set_text_props(fontweight="bold", color=INK); cell.set_facecolor("#f2f2f2")
    elif c_ >= 0 and best_of[cols_t[c_]] == tab.index[r_ - 1]:
        cell.set_text_props(fontweight="bold", color=GREEN if cols_t[c_].startswith("R²") else INK)
        cell.set_facecolor("#e8f5f2")
ax.set_title("All scenarios at a glance — best value per column highlighted", pad=8, color=INK)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_recap_table.png", dpi=150,
                                bbox_inches="tight"); plt.show()
tab.round(3)

In [ ]:
# === Recap (c): quality vs cost — one point per (model, encoding, X) ===
# The scatter that closes the model-choice argument: the winner must sit top-left (best R², low training
# cost). RF positional at X=10 sits bottom-right: the most expensive configuration is also the worst.
from matplotlib.lines import Line2D

MK = {"none": "D", "pos": "o", "agg": "s"}
fig, ax = plt.subplots(figsize=(8.5, 5))
for _, r_ in res.iterrows():
    color = GREEN if r_.model == "NN" else RED
    ax.scatter(r_.train_s, r_.R2, s=90, color=color, marker=MK.get(r_.enc, "o"),
               edgecolor="white", lw=1, zorder=3)
best_row = res.loc[res.R2.idxmax()]
ax.annotate(f"best: {best_row.model} X={int(best_row.X)} {best_row.enc}\nR²={best_row.R2:.3f}",
            (best_row.train_s, best_row.R2), xytext=(best_row.train_s + 25, best_row.R2 - .004),
            color=INK, fontweight="bold", arrowprops=dict(arrowstyle="-", color=MUT))
worst = res.loc[res.R2.idxmin()]
ax.annotate(f"{worst.model} X={int(worst.X)} {worst.enc}:\nmost costly AND worst",
            (worst.train_s, worst.R2), xytext=(worst.train_s - 140, worst.R2 + .004),
            color=MUT, fontsize=9, arrowprops=dict(arrowstyle="-", color=MUT))
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=GREEN, label="NN"),
                   Line2D([], [], marker="o", ls="", color=RED, label="RF"),
                   Line2D([], [], marker="D", ls="", color=MUT, label="X=0 baseline"),
                   Line2D([], [], marker="o", ls="", color=MUT, label="positional"),
                   Line2D([], [], marker="s", ls="", color=MUT, label="aggregated")],
          frameon=False, ncol=2, fontsize=9)
ax.set_xlabel("training time (s)"); ax.set_ylabel("R² (test)")
ax.set_title("Quality vs cost — where every configuration lands")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=.25); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_quality_vs_cost.png", dpi=150); plt.show()

In [ ]:
# === Recap (d): NN vs RF per scenario — who wins, and by how much ===
# Dumbbell chart: one row per scenario, a connector between the two models' R². The pattern IS the
# story: RF wins on clean features (baseline, aggregated), the NN resists the noisy positional columns
# better at large X — and no configuration beats RF + aggregated.
from matplotlib.lines import Line2D

scen_r = [("X=0 baseline", res[res.enc == "none"])]
for x_ in sorted(res.X[res.X > 0].unique()):
    for enc_, lbl in [("pos", "positional"), ("agg", "aggregated")]:
        d_ = res[(res.X == x_) & (res.enc == enc_)]
        if len(d_): scen_r.append((f"X={x_} {lbl}", d_))

fig, ax = plt.subplots(figsize=(8.5, 0.62 * len(scen_r) + 1.6))
ypos = np.arange(len(scen_r))[::-1]
for y, (name, d_) in zip(ypos, scen_r):
    nn = d_.loc[d_.model == "NN", "R2"]; rf = d_.loc[d_.model == "RF", "R2"]
    if not len(nn) or not len(rf):
        continue
    nn, rf = float(nn.iloc[0]), float(rf.iloc[0])
    ax.plot([nn, rf], [y, y], color=MUT, lw=1.6, zorder=2)
    ax.scatter([nn], [y], s=80, color=GREEN, zorder=3, edgecolor="white", lw=1)
    ax.scatter([rf], [y], s=80, color=RED, zorder=3, edgecolor="white", lw=1)
    win = "RF" if rf > nn else "NN"
    ax.text(max(nn, rf) + .0035, y, f"{win} +{abs(rf - nn):.3f}", va="center", fontsize=8.5,
            color=RED if win == "RF" else GREEN, fontweight="bold")
ax.set_yticks(ypos); ax.set_yticklabels([n for n, _ in scen_r])
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=GREEN, label="NN"),
                   Line2D([], [], marker="o", ls="", color=RED, label="RF")],
          frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(.5, 1.0))
ax.set_xlabel("R² (test)")
ax.set_title("NN vs RF per scenario — who wins, and by how much", pad=30)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=.25); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_nn_vs_rf_dumbbell.png", dpi=150); plt.show()

**Reading notes for the recap figures.**
- **Feature anatomy:** every scenario shares the same 13 base features (7 own numeric + 6 traffic one-hot);
  the encodings differ only in how the X closest users enter: 5 columns *per neighbour* (positional) vs 5
  order-invariant aggregates *in total*. This is also why the TL notebook freezes `BEST_X`/`BEST_ENC`: the
  pretrained NN's input layer has a fixed width, so the Salt&Tar matrix must match the ACC schema column by
  column (same features, same order, same ACC-fitted scaler, all 6 one-hot classes present) — changing X or
  the encoding would invalidate the pretrained weights entirely.
- **NN vs RF:** RF wins wherever the features are clean (baseline and aggregated: a piecewise-constant,
  quantized demand target suits trees, with minimal tuning), while the NN is the more robust of the two under
  the *positional* encoding at large X — with `max_features="sqrt"` the forest is forced to split on
  near-duplicate noisy neighbour columns, whereas the network spreads small weights across them. Nobody beats
  RF + aggregated.
- **Model selection metric:** on a fixed test set R² is a monotone transform of MSE, so ranking by R² is
  legitimate — but MAE is always shown next to it because squared-error metrics over-weight the heavy tail
  (EDA: the top 1% of samples carries ~86% of the variance before the p99 cut). Here R² and MAE agree on the
  winner (RF aggregated), which makes the selection robust; train/inference time act as tie-breakers.